# Medical Question Answering — Projet académique

Ce notebook implémente un système de **Medical Q&A** en comparant plusieurs stratégies :
- **Stratégie 1** : Zero-shot (Direct + Prompt Engineering)
- **Stratégie 2** : RAG (Retrieval Augmented Generation)
- **Stratégie 3** : Fine-tuning QLoRA

**Environnement** : Google Colab (GPU T4), PyTorch, Transformers, PEFT, TRL, Evaluate, LangChain.

---
## Étape 1 : Installation et préparation

On installe les dépendances, charge le dataset **medalpaca/medical_meadow_medical_flashcards**, l’analyse succinctement et crée un split **90 % train / 10 % test** (pas de split fourni par le dataset).

### 1.1 Installation des dépendances

Exécuter cette cellule en premier (idéalement avec **GPU T4** activé dans Colab : *Runtime > Change runtime type*).

In [1]:
# Installation des bibliothèques pour Colab (GPU T4)
!pip install -q "torch>=2.0" "transformers>=4.36" "accelerate>=0.25" "bitsandbytes>=0.41"
!pip install -q "peft>=0.7" "trl>=0.7" "datasets>=2.14"
!pip install -q "evaluate>=0.4" "rouge_score>=0.1"
!pip install -q "langchain>=0.1" "langchain-community>=0.0.10" "langchain-core"
!pip install -q "sentence-transformers>=2.2"
# FAISS : faiss-gpu sur Colab avec GPU, sinon faiss-cpu
!pip install -q faiss-gpu 2>/dev/null || !pip install -q faiss-cpu

print("Installation terminée.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.9/532.9 kB 13.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
/bin/bash: line 1: !pip: command not found
Installation terminée.


### 1.2 Imports et configuration

In [2]:
import os
import gc
import torch
from datasets import load_dataset
import pandas as pd

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
GPU: Tesla T4


### 1.3 Chargement et analyse du dataset

Le dataset fournit des paires **Question / Answer** (colonnes `input` et `output`). Il n’y a pas de split ; on utilisera `train_test_split` pour créer train (90 %) et test (10 %).

In [3]:
DATASET_NAME = "medalpaca/medical_meadow_medical_flashcards"

dataset = load_dataset(DATASET_NAME)
print("Clés:", dataset.keys())
print("Structure:", dataset)

# Le dataset a souvent une clé "train"
if "train" in dataset:
    full = dataset["train"]
else:
    full = dataset[list(dataset.keys())[0]]

print("\nColonnes:", full.column_names)
print("Nombre d'exemples:", len(full))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

medical_meadow_wikidoc_medical_flashcard(…):   0%|          | 0.00/17.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/33955 [00:00<?, ? examples/s]

Clés: dict_keys(['train'])
Structure: DatasetDict({
    train: Dataset({
        features: ['input', 'output', 'instruction'],
        num_rows: 33955
    })
})

Colonnes: ['input', 'output', 'instruction']
Nombre d'exemples: 33955


### 1.4 Exemples et normalisation des colonnes

On s’assure d’avoir des champs `question` et `answer` (en mappant `input` → `question`, `output` → `answer` si nécessaire).

In [4]:
# Normalisation : input -> question, output -> answer (medalpaca)
data = full.rename_columns({"input": "question", "output": "answer"})
data = data.remove_columns([c for c in data.column_names if c not in ["question", "answer"]])

# 2–3 exemples
for i in range(min(3, len(data))):
    r = data[i]
    print(f"--- Exemple {i+1} ---\nQuestion: {(r['question'][:180] + '...') if len(r['question']) > 180 else r['question']}\nAnswer: {(r['answer'][:180] + '...') if len(r['answer']) > 180 else r['answer']}\n")

--- Exemple 1 ---
Question: What is the relationship between very low Mg2+ levels, PTH levels, and Ca2+ levels?
Answer: Very low Mg2+ levels correspond to low PTH levels which in turn results in low Ca2+ levels.

--- Exemple 2 ---
Question: What leads to genitourinary syndrome of menopause (atrophic vaginitis)?
Answer: Low estradiol production leads to genitourinary syndrome of menopause (atrophic vaginitis).

--- Exemple 3 ---
Question: What does low REM sleep latency and experiencing hallucinations/sleep paralysis suggest?
Answer: Low REM sleep latency and experiencing hallucinations/sleep paralysis suggests narcolepsy.



In [5]:
### 1.5 Split 90 % train / 10 % test

Split aléatoire avec `train_test_split` (seed pour reproductibilité). Le **test** servira à l’évaluation des stratégies.

SyntaxError: invalid character '’' (U+2019) (ipython-input-1886639178.py, line 3)

In [ ]:
# 90 % train, 10 % test (seed=42)
splits = data.train_test_split(test_size=0.10, seed=42)
train_data = splits["train"]
test_data = splits["test"]
print(f"Train: {len(train_data)} | Test: {len(test_data)}")

---
## Étape 2 : Stratégie 1 — Zero-shot et Prompt Engineering

On charge un **modèle léger en 4-bit** (TinyLlama ou Zephyr) et on définit `generate_answer(question, strategy="direct"|"prompt_engineering")` :
- **Direct** : seule la question est envoyée au modèle.
- **Prompt Engineering** : un *system prompt* du type *"You are a medical expert. Answer concisely."* est ajouté.

In [ ]:
# Libérer la mémoire GPU avant de charger un nouveau modèle
def _clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Modèle léger compatible Colab T4 (TinyLlama 1.1B) en 4-bit
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb, device_map="auto")
model.eval()
print("Modèle chargé (4-bit).")

In [ ]:
def generate_answer(question, strategy="direct", model=model, tokenizer=tokenizer, context=None, max_new_tokens=80):
    """
    strategy: "direct" | "prompt_engineering" | "rag"
    - direct: seule la question.
    - prompt_engineering: system prompt "You are a medical expert. Answer concisely."
    - rag: context doit être fourni; on l'ajoute au user message.
    """
    if strategy == "direct":
        messages = [{"role": "user", "content": question}]
    elif strategy == "prompt_engineering":
        content = "You are a medical expert. Answer the following question concisely in one or two sentences.\n\n" + question
        messages = [{"role": "user", "content": content}]
    elif strategy == "rag" and context:
        user = f"Use the following context to answer.\n\nContext:\n{context}\n\nQuestion: {question}"
        messages = [{"role": "user", "content": user}]
    else:
        messages = [{"role": "user", "content": question}]

    inp = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True)
    if not isinstance(inp, dict):
        inp = {"input_ids": inp}
    inp = {k: v.to(model.device) for k, v in inp.items()}
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)

    text = tokenizer.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    return text

In [ ]:
---
## Étape 3 : Stratégie 2 — RAG (Retrieval Augmented Generation)

Pipeline RAG avec **LangChain** : embeddings **SentenceTransformer**, base vectorielle **FAISS**.
On indexe une **petite partie du train** comme base de connaissance (ou Wikipedia si on le souhaite).
La génération récupère le contexte pertinent puis appelle le LLM.

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document

# Embeddings : SentenceTransformer via HuggingFace
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Sous-ensemble du train comme base de connaissance (ex. 800 ex.)
n_rag = min(800, len(train_data))
docs = [
    Document(page_content=train_data[i]["question"], metadata={"answer": train_data[i]["answer"]})
    for i in range(n_rag)
]
vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("Index FAISS construit, k=3.")

In [ ]:
def get_rag_context(question, retriever=retriever):
    hits = retriever.get_relevant_documents(question)
    parts = [f"Q: {d.page_content} A: {d.metadata.get('answer', '')}" for d in hits]
    return "\n\n".join(parts)

def generate_answer_rag(question, model=model, tokenizer=tokenizer):
    context = get_rag_context(question)
    return generate_answer(question, strategy="rag", context=context, model=model, tokenizer=tokenizer)

---
## Étape 4 : Stratégie 3 — Fine-tuning QLoRA

Fine-tuning avec **PEFT (LoRA)** et **SFTTrainer (TRL)** en 4-bit (QLoRA).  
Réglage volontairement **léger** : 1 epoch, `max_steps` limité (~50) pour rester sous ~15 min sur Colab.  
Sauvegarde de l’**adaptateur LoRA** à la fin.

In [ ]:
# Libérer le modèle d'inférence pour faire de la place au fine-tuning
del model
_clear_cuda()

from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Recharger le modèle en 4-bit pour l'entraînement
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
model_ft = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb, device_map="auto")
model_ft = prepare_model_for_kbit_training(model_ft)

peft_config = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM")

def formatting_prompts_func(ex):
    # Format type ChatML (compatible TinyLlama, SmolLM, etc.)
    return f"<|im_start|>user\n{ex['question']}<|im_end|>\n<|im_start|>assistant\n{ex['answer']}<|im_end|>"

# Sous-ensemble d'entraînement pour aller vite (~100 ex)
n_train_ft = min(100, len(train_data))
train_ft = train_data.select(range(n_train_ft))

sft_config = SFTConfig(
    output_dir="./lora_medical_qa",
    num_train_epochs=1,
    max_steps=50,
    max_length=256,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    fp16=False,
    bf16=torch.cuda.is_available(),
    logging_steps=10,
    save_steps=50,
    save_total_limit=1,
)

trainer = SFTTrainer(
    model=model_ft,
    args=sft_config,
    train_dataset=train_ft,
    formatting_func=formatting_prompts_func,
    peft_config=peft_config,
)
trainer.train()
trainer.save_model("./lora_medical_qa")
print("Adaptateur LoRA sauvegardé dans ./lora_medical_qa")

---
## Étape 5 : Évaluation et comparaison

On évalue sur un **sous-ensemble de 20 exemples du test**.  
- **ROUGE** : similarité syntaxique entre prédiction et réponse de référence.  
- **LLM-as-a-judge** : le modèle note de 1 à 5 la pertinence de la réponse par rapport à la référence.  
Tableau final : scores moyens pour **Direct**, **Prompt**, **RAG**, **Fine-tuned**.

In [ ]:
import evaluate

# Sous-ensemble de test (20 ex.) pour évaluation rapide
N_EVAL = 20
eval_data = test_data.select(range(min(N_EVAL, len(test_data))))
questions = [eval_data[i]["question"] for i in range(len(eval_data))]
gold = [eval_data[i]["answer"] for i in range(len(eval_data))]

def compute_rouge(pred, ref):
    r = evaluate.load("rouge")
    res = r.compute(predictions=[pred], references=[ref])
    def _v(k):
        x = res.get(k)
        if x is None: return 0.0
        return float(x[0]) if isinstance(x, list) else float(x)
    return {"rouge1": _v("rouge1"), "rouge2": _v("rouge2"), "rougeL": _v("rougeL")}

def llm_as_judge(question, gold_answer, pred_answer, model, tokenizer, max_new_tokens=20):
    """Note de 1 à 5 : pertinence de pred par rapport à gold."""
    prompt = f"On 1-5, rate how relevant this answer is to the reference (1=not relevant, 5=very relevant). Only output the number.\nRef: {gold_answer[:200]}\nPred: {pred_answer[:200]}\nScore:"
    messages = [{"role": "user", "content": prompt}]
    inp = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True)
    if not isinstance(inp, dict):
        inp = {"input_ids": inp}
    inp = {k: v.to(model.device) for k, v in inp.items()}
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    text = tokenizer.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    for c in text:
        if c.isdigit():
            return min(5, max(1, int(c)))
    return 3

In [ ]:
# Libérer la mémoire et recharger le modèle de base pour Direct, Prompt, RAG
try: del model_ft, trainer
except NameError: pass
_clear_cuda()

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb, device_map="auto")
model.eval()

results = { "Direct": {"rouge1": [], "rouge2": [], "rougeL": [], "judge": []},
            "Prompt": {"rouge1": [], "rouge2": [], "rougeL": [], "judge": []},
            "RAG":     {"rouge1": [], "rouge2": [], "rougeL": [], "judge": []} }

for i in range(len(questions)):
    q, g = questions[i], gold[i]
    for strat, key in [("direct", "Direct"), ("prompt_engineering", "Prompt")]:
        pred = generate_answer(q, strategy=strat, model=model, tokenizer=tokenizer)
        r = compute_rouge(pred, g)
        results[key]["rouge1"].append(r["rouge1"] or 0)
        results[key]["rouge2"].append(r["rouge2"] or 0)
        results[key]["rougeL"].append(r["rougeL"] or 0)
        results[key]["judge"].append(llm_as_judge(q, g, pred, model, tokenizer))
    pred_rag = generate_answer_rag(q, model=model, tokenizer=tokenizer)
    r = compute_rouge(pred_rag, g)
    results["RAG"]["rouge1"].append(r["rouge1"] or 0)
    results["RAG"]["rouge2"].append(r["rouge2"] or 0)
    results["RAG"]["rougeL"].append(r["rougeL"] or 0)
    results["RAG"]["judge"].append(llm_as_judge(q, g, pred_rag, model, tokenizer))

print("Direct, Prompt, RAG évalués.")

In [ ]:
# Évaluation Fine-tuned : recharger base + adaptateur LoRA
try: del model
except NameError: pass
_clear_cuda()

from peft import PeftModel

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb, device_map="auto")
model = PeftModel.from_pretrained(model, "./lora_medical_qa")
model.eval()

results["Fine-tuned"] = {"rouge1": [], "rouge2": [], "rougeL": [], "judge": []}
for i in range(len(questions)):
    q, g = questions[i], gold[i]
    pred = generate_answer(q, strategy="prompt_engineering", model=model, tokenizer=tokenizer)
    r = compute_rouge(pred, g)
    results["Fine-tuned"]["rouge1"].append(r["rouge1"] or 0)
    results["Fine-tuned"]["rouge2"].append(r["rouge2"] or 0)
    results["Fine-tuned"]["rougeL"].append(r["rougeL"] or 0)
    results["Fine-tuned"]["judge"].append(llm_as_judge(q, g, pred, model, tokenizer))

print("Fine-tuned évalué.")

In [ ]:
# Tableau comparatif : moyennes par stratégie
rows = []
for name in ["Direct", "Prompt", "RAG", "Fine-tuned"]:
    r = results[name]
    rows.append({
        "Stratégie": name,
        "ROUGE-1": pd.Series(r["rouge1"]).mean(),
        "ROUGE-2": pd.Series(r["rouge2"]).mean(),
        "ROUGE-L": pd.Series(r["rougeL"]).mean(),
        "LLM-judge (1-5)": pd.Series(r["judge"]).mean(),
    })
df = pd.DataFrame(rows)
display(df)

### Interprétation

- **ROUGE** mesure le chevauchement lexical (n-grammes) entre la prédiction et la référence ; des scores plus élevés indiquent une similarité de surface plus forte.
- **LLM-as-a-judge** donne une note de pertinence sémantique (1–5) ; il est utile pour comparer des formulations différentes qui restent correctes.
- Le **fine-tuning** et le **RAG** ont souvent un avantage sur le zero-shot pour des questions médicales spécialisées, sous réserve de suffisamment de données (RAG) ou d’un entraînement adapté (QLoRA).